In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import kagglehub


### Step 1: Download ORL Dataset

In [ ]:
# Download latest version
path = kagglehub.dataset_download("kasikrit/att-database-of-faces")
print("Path to dataset files:", path)


Path to dataset files: /home/habiba/.cache/kagglehub/datasets/kasikrit/att-database-of-faces/versions/2


### Step 2: Generate the Data Matrix and the Label vector

In the AT&T Face Dataset, a subject refers to a person. The dataset includes 40 different people, and each one is stored in a separate folder
Reading all 400 images (10 per person × 40 people)
Storing them in images (shape: 400, 112, 92)
Assigning a label (person ID) from 1 to 40 to each image

In [19]:
images = []
Y = []
# Loop through subjects (s1 to s40)
for subject_id in range(1, 41): 
    subject_folder = os.path.join(path, f's{subject_id}')
    
    # Loop through each of the 10 images for the subject
    for img_number in range(1, 11):  # 1.pgm to 10.pgm
        img_path = os.path.join(subject_folder, f'{img_number}.pgm')
        
        # Read image 
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Append image and label
        images.append(img)
        Y.append(subject_id)  # Use subject_id as the label

images = np.array(images)        
Y = np.array(Y) 
D = images.reshape(400, -1)
D.shape

print("Loaded dataset shape:", D.shape)
print("Y shape:", Y.shape)


Loaded dataset shape: (400, 10304)
Y shape: (400,)


### Step 3: Split the Dataset into Training and Test sets

In [91]:
# Select training and testing rows
D_train = D[::2]  # odd-numbered rows
D_test  = D[1::2] # even-numbered rows 

y_train = Y[::2]
y_test  = Y[1::2]

### PCA Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.offsetbox as offsetbox

class PCA:
    def __init__(self, D_train, D_test, y_train, y_test):
        """Class intialization"""
        self.D_train = D_train
        self.D_test = D_test
        self.y_train = y_train
        self.y_test = y_test
        self.D_centered = []
        self.eigenvalues = []
        self.eigenvectors = []

    def compCov(self):
        """Compute the covariance matrix"""
        mean_face = np.mean(self.D_train, axis=0)
        self.D_centered = self.D_train - mean_face
        Cov = self.D_centered @ self.D_centered.T
        Cov = Cov / self.D_train.shape[0]
        return Cov

    def compEig(self, Cov):
        """Compute the eigenvalues and the eigenvectors in feature space"""
        eigenvalues, eigenvectors = np.linalg.eigh(Cov)
        eigenvalues = eigenvalues[::-1]
        eigenvectors = eigenvectors[:, ::-1]

        # Convert to feature space
        eigvecs_feat_space = self.D_centered.T @ eigenvectors 
        eigvecs_feat_space = eigvecs_feat_space[:, :len(eigenvalues)]  # reduce to same k

        norm_eigenvect = eigvecs_feat_space / np.linalg.norm(eigvecs_feat_space, axis=0)

        self.eigenvalues = eigenvalues
        self.eigenvectors = norm_eigenvect


    def selectComp(self, alpha):
        """Select the first i eigenvectors based on the explained variance"""
        sum = 0
        eigsum = np.sum(self.eigenvalues)
        for i, val in enumerate(self.eigenvalues):
            sum += val
            if sum / eigsum >= alpha:
                return self.eigenvectors[:, :i+1]

    def project(self, alpha):
        """Project the data on the eigenvectors"""
        Cov = self.compCov()
        self.compEig(Cov)
        eigvec = self.selectComp(alpha)
        proj = self.D_centered @ eigvec
        return proj , eigvec

    def visualize_projection(self, alpha, image_shape=(112, 92)):
        """Image reconstruction"""
        projected , subeigvec = self.project(alpha)
        recon = projected[22] @ subeigvec.T
        mean_face = np.mean(self.D_train, axis=0)
        recon_image = recon + mean_face
        recon_image = recon_image.reshape(image_shape)

        plt.imshow(recon_image, cmap='gray')
        plt.title(f"Reconstructed Image with alpha={alpha} and Dimensions ={projected.shape[1]}")
        plt.axis('off')
        plt.show()


pca = PCA(D_train, D_test, y_train, y_test)
pca.visualize_projection(alpha=0.98)
pca.visualize_projection(alpha=0.85)
pca.visualize_projection(alpha=0.9)
pca.visualize_projection(alpha=0.8)
pca.visualize_projection(alpha=0.5)


## Unsupervised Clustering
### K-Means Clustering

### K-Means Clustering Evaluation

### Gaussian Mixture Model Clustering

### Gaussian Mixture Model Clustering Evaluation

### Bonus